# Chapter 9 · Reinforcement Learning and Agents
**Mastering Agentic AI** — Manning Publications



## What this notebook covers
- RL foundations: state, action, reward in the Diet Coach context
- The reward function: designing signals that align with user outcomes
- RLHF-style preference collection: generating pairs and collecting labels
- Multi-Agent RL (MARL): cooperative reward shaping
- BanditPromptOptimiser: UCB1 exploration over prompt variants
- The RLHF production pipeline sketch

---

## 9.0 · Setup

In [ ]:
# !pip install anthropic python-dotenv
import os, json, random, statistics, math
from dataclasses import dataclass, field
from typing import Any
from anthropic import Anthropic
from dotenv import load_dotenv
load_dotenv()
assert os.getenv("ANTHROPIC_API_KEY"), "See appendix_a_api_keys.md"
client = Anthropic()
MODEL = "claude-opus-4-5"
print("Ready")

## 9.1 · RL Foundations — State, Action, Reward

In [ ]:
@dataclass
class DietCoachState:
    """
    State representation for the Diet Coach RL environment.
    
    State = what the agent knows at decision time.
    Good state design satisfies the Markov property: the agent can
    choose the optimal action from the state alone, without the full history.
    """
    user_goal:          str    # "lose 5 kg"
    days_on_plan:       int
    current_streak:     int    # consecutive days meeting goal
    protein_deficit_g:  float  # today's shortfall vs target
    last_mood:          str    # "motivated" | "struggling" | "neutral"
    milestones_hit:     list[str] = field(default_factory=list)

@dataclass
class CoachAction:
    """Actions available to the Diet Coach agent."""
    action_type: str    # "encourage" | "correct" | "challenge" | "celebrate" | "refer"
    message:     str

def reward(state: DietCoachState, action: CoachAction) -> float:
    """
    Section 9.1: Reward function — maps (state, action) → scalar.
    
    Design principle: reward the OUTCOME we care about (user adherence,
    wellbeing), not the proxy (user says 'thanks').
    
    In production: combine human preference labels with automated signals
    (meal logging frequency, self-reported energy levels).
    """
    r = 0.0
    # Appropriate action given mood
    if state.last_mood == "struggling" and action.action_type == "encourage": r += 0.5
    if state.last_mood == "motivated"  and action.action_type == "challenge":  r += 0.4
    if state.last_mood == "struggling" and action.action_type == "correct":    r -= 0.3
    # Nutritional targeting
    if state.protein_deficit_g > 20 and "protein" in action.message.lower():  r += 0.3
    # Streak recognition
    if state.current_streak >= 7 and action.action_type == "celebrate":         r += 0.5
    # Safety language always good
    if "professional" in action.message.lower() or "dietitian" in action.message.lower(): r += 0.1
    return max(-1.0, min(1.0, r))   # clip to [-1, 1]

# Test the reward function
state = DietCoachState("lose 5kg", days_on_plan=14, current_streak=7,
                       protein_deficit_g=5, last_mood="motivated")
action_good = CoachAction("celebrate", "Amazing — 7 days straight! You're building a real habit.")
action_bad  = CoachAction("correct", "You need to eat more protein.")

print(f"Good action reward: {reward(state, action_good):.1f}")
print(f"Bad action reward:  {reward(state, action_bad):.1f}")
print()
print("Key insight: 'correct' when the user is motivated is a missed opportunity.")
print("The reward function encodes this domain knowledge explicitly.")

## 9.2 · RLHF-Style Preference Collection

In [ ]:
@dataclass
class PreferencePair:
    """A human-preference data point: two responses, one preferred."""
    prompt:     str
    response_a: str
    response_b: str
    preferred:  str        # "A" or "B"
    reason:     str | None = None

class PreferenceCollector:
    """
    Section 9.2: Collects preference data for RLHF fine-tuning.
    
    Workflow:
      1. Generate two responses (different temperatures / prompts)
      2. LLM judge selects the preferred response
      3. Store preference pairs as JSONL for reward model training
      
    Steps 4-5 (reward model + PPO/GRPO fine-tuning) require a training
    cluster and are out of scope for a notebook environment.
    """
    def __init__(self):
        self.pairs: list[PreferencePair] = []

    def generate_pair(self, prompt: str, system: str) -> tuple[str, str]:
        """Generate two responses: one conservative, one exploratory."""
        def call(temp_instruction: str) -> str:
            r = client.messages.create(
                model=MODEL, max_tokens=256,
                system=system + f"\n\nStyle: {temp_instruction}",
                messages=[{"role": "user", "content": prompt}]
            )
            return r.content[0].text
        a = call("Be precise and evidence-based.")
        b = call("Be warm, motivational, and conversational.")
        return a, b

    def judge_pair(self, prompt: str, response_a: str, response_b: str,
                   criteria: str) -> tuple[str, str]:
        """Use LLM to select the better response."""
        judge_prompt = (
            f"Which response better meets this criteria: {criteria}\n\n"
            f"User prompt: {prompt}\n\n"
            f"Response A: {response_a}\n\n"
            f"Response B: {response_b}\n\n"
            "Reply with ONLY: 'A' or 'B' followed by one sentence reason."
        )
        r = client.messages.create(model=MODEL, max_tokens=60,
            messages=[{"role": "user", "content": judge_prompt}])
        result = r.content[0].text.strip()
        preferred = "A" if result.startswith("A") else "B"
        return preferred, result

    def collect(self, prompt: str, criteria: str) -> PreferencePair:
        system = "You are an AI Diet Coach."
        a, b = self.generate_pair(prompt, system)
        preferred, reason = self.judge_pair(prompt, a, b, criteria)
        pair = PreferencePair(prompt=prompt, response_a=a, response_b=b,
                              preferred=preferred, reason=reason)
        self.pairs.append(pair)
        return pair

    def export_jsonl(self) -> str:
        """Export as JSONL for reward model training."""
        lines = [json.dumps({"prompt": p.prompt,
                             "chosen":  p.response_a if p.preferred == "A" else p.response_b,
                             "rejected": p.response_b if p.preferred == "A" else p.response_a})
                 for p in self.pairs]
        return "\n".join(lines)

collector = PreferenceCollector()
pair = collector.collect(
    prompt="I ate pizza and chips for dinner. How bad is that?",
    criteria="Supportive tone, avoids guilt, provides actionable next step"
)
print(f"Preferred: {pair.preferred}")
print(f"Reason: {pair.reason}")
print(f"\nChosen response preview: {(pair.response_a if pair.preferred=='A' else pair.response_b)[:200]}")

## 9.3 · BanditPromptOptimiser — UCB1 Exploration

In [ ]:
class BanditPromptOptimiser:
    """
    Section 9.4: Multi-armed bandit for prompt variant selection.
    
    Uses UCB1 (Upper Confidence Bound) to balance:
      - Exploitation: use the best-performing prompt seen so far
      - Exploration:  try less-used variants to find better ones
    
    This is a lightweight alternative to full DSPy optimisation when
    you have a few discrete prompt variants and live user feedback signals.
    """
    def __init__(self, variants: list[dict]):
        self.variants = variants
        self.counts   = [0] * len(variants)    # times each variant was used
        self.rewards  = [0.0] * len(variants)  # cumulative reward per variant

    def select(self) -> int:
        """UCB1: select variant with highest upper confidence bound."""
        total = sum(self.counts)
        if total == 0:
            return 0
        ucb_scores = []
        for i, (count, reward_sum) in enumerate(zip(self.counts, self.rewards)):
            if count == 0:
                return i   # always try untested variants first
            mean = reward_sum / count
            confidence = math.sqrt(2 * math.log(total) / count)
            ucb_scores.append(mean + confidence)
        return ucb_scores.index(max(ucb_scores))

    def update(self, variant_idx: int, reward: float) -> None:
        self.counts[variant_idx]  += 1
        self.rewards[variant_idx] += reward

    def best_variant(self) -> dict:
        if not any(self.counts):
            return self.variants[0]
        avg_rewards = [r / c if c > 0 else 0 for r, c in zip(self.rewards, self.counts)]
        return self.variants[avg_rewards.index(max(avg_rewards))]

PROMPT_VARIANTS = [
    {"name": "clinical",      "style": "Be precise, cite specific numbers, avoid small talk."},
    {"name": "warm",          "style": "Be warm and encouraging. Celebrate small wins."},
    {"name": "socratic",      "style": "Ask reflective questions to help the user discover insights."},
    {"name": "accountability","style": "Be direct and hold the user accountable to their stated goals."},
]

optimiser = BanditPromptOptimiser(PROMPT_VARIANTS)

# Simulate 20 rounds of feedback
for _ in range(20):
    idx = optimiser.select()
    # Simulate reward: warm and socratic tend to perform better (simulated)
    simulated_reward = {"clinical": 0.5, "warm": 0.8, "socratic": 0.75, "accountability": 0.6}
    r = simulated_reward[PROMPT_VARIANTS[idx]["name"]] + random.gauss(0, 0.1)
    optimiser.update(idx, max(0, min(1, r)))

print("Bandit results after 20 rounds:")
for i, v in enumerate(PROMPT_VARIANTS):
    avg = optimiser.rewards[i] / optimiser.counts[i] if optimiser.counts[i] > 0 else 0
    print(f"  {v['name']:15s}: used {optimiser.counts[i]:2d}x | avg reward {avg:.2f}")
print(f"\nBest variant: {optimiser.best_variant()['name']}")

## 9.4 · Chapter Summary

| Concept | What we built |
|---|---|
| DietCoachState / CoachAction | RL state and action spaces for the Diet Coach |
| reward() | Explicit reward function encoding domain knowledge |
| PreferenceCollector | RLHF data collection: generate pairs → judge → export JSONL |
| BanditPromptOptimiser | UCB1 exploration over discrete prompt variants |

**Production RLHF pipeline (beyond this notebook):**
1. Collect preference pairs at scale (human raters or strong LLM judge)
2. Train a reward model on preference pairs (Bradley-Terry or similar)
3. Fine-tune the policy with PPO or GRPO using the reward model
4. Deploy, monitor for reward hacking, retrain periodically

**What is next?** Chapter 10 — Security and Governance: making the Diet Coach production-safe.

---
*Mastering Agentic AI · Manning Publications*